In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
#from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
from torchvision import transforms
import torchvision
#from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
cudnn.benchmark = True
plt.ion()

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
])

# #Testing 
# img = transform(Image.open("data/01/AS.png").convert("RGB"))
# print(img.shape)
# torch.set_printoptions(profile="full")
# print(img)


In [2]:
import os
import shutil
import random

# Create validation dataset by copying 10 images from each card folder
val_base_dir = '/home/sawyer/card-counting/val'
train_base_dir = '/home/sawyer/card-counting/data'

# Remove existing val directory if it exists
if os.path.exists(val_base_dir):
    shutil.rmtree(val_base_dir)

# Create val directory structure and copy images
for card_num in range(1, 53):
    card_folder = f'{card_num:02d}'
    source_dir = os.path.join(train_base_dir, card_folder)
    val_dir = os.path.join(val_base_dir, card_folder)
    
    # Create validation folder for this card
    os.makedirs(val_dir, exist_ok=True)
    
    # Get all images from the source folder
    if os.path.exists(source_dir):
        all_images = [f for f in os.listdir(source_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
        # Randomly select 10 images
        selected_images = random.sample(all_images, min(10, len(all_images)))
        
        # Copy selected images
        for image in selected_images:
            src_path = os.path.join(source_dir, image)
            dst_path = os.path.join(val_dir, image)
            shutil.copy2(src_path, dst_path)


print(f'\nValidation dataset created at {val_base_dir}')



Validation dataset created at /home/sawyer/card-counting/val


In [3]:
batch_size = 64

trainset = torchvision.datasets.ImageFolder(root='/home/sawyer/card-counting/data', transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

classes = ( 
    '01', '02', '03', '04', '05', '06', '07', '08', '09', '10',
    '11', '12', '13', '14', '15', '16', '17', '18', '19', '20',
    '21', '22', '23', '24', '25', '26', '27', '28', '29', '30',
    '31', '32', '33', '34', '35', '36', '37', '38', '39', '40',
    '41', '42', '43', '44', '45', '46', '47', '48', '49', '50',
    '51', '52'
)


In [4]:
valset = torchvision.datasets.ImageFolder(root='/home/sawyer/card-counting/val', transform=transform)
valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                        shuffle=False, num_workers=2)

print(f'Validation set size: {len(valset)} images')
print(f'Training set size: {len(trainset)} images')


Validation set size: 520 images
Training set size: 2758 images


In [5]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=52):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),     # 128x128

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),     # 64x64

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),     # 32x32

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),     # 16x16
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 * 16 * 16, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

net = SimpleCNN(num_classes=52).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    print(f"\n----- Epoch {epoch+1}/{epochs} -----")

    # TRAINING -----------------------------
    net.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Train Loss: {train_loss/len(trainloader):.4f} | Train Acc: {train_acc:.2f}%")
    net.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in valloader:
            images, labels = images.to(device), labels.to(device)

            outputs = net(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"Val Loss: {val_loss/len(valloader):.4f} | Val Acc: {val_acc:.2f}%")

print("\nTraining Finished.")
torch.save(net.state_dict(), "card_model.pth")
print("Model saved as card_model.pth")


Using: cuda

----- Epoch 1/10 -----

----- Epoch 1/10 -----
Train Loss: 3.1355 | Train Acc: 19.22%
Train Loss: 3.1355 | Train Acc: 19.22%
Val Loss: 1.5129 | Val Acc: 57.31%

----- Epoch 2/10 -----
Val Loss: 1.5129 | Val Acc: 57.31%

----- Epoch 2/10 -----
Train Loss: 1.3717 | Train Acc: 57.36%
Train Loss: 1.3717 | Train Acc: 57.36%
Val Loss: 0.4940 | Val Acc: 84.81%

----- Epoch 3/10 -----
Val Loss: 0.4940 | Val Acc: 84.81%

----- Epoch 3/10 -----
Train Loss: 0.6894 | Train Acc: 78.68%
Train Loss: 0.6894 | Train Acc: 78.68%
Val Loss: 0.2103 | Val Acc: 94.04%

----- Epoch 4/10 -----
Val Loss: 0.2103 | Val Acc: 94.04%

----- Epoch 4/10 -----
Train Loss: 0.3999 | Train Acc: 87.60%
Train Loss: 0.3999 | Train Acc: 87.60%
Val Loss: 0.1169 | Val Acc: 96.73%

----- Epoch 5/10 -----
Val Loss: 0.1169 | Val Acc: 96.73%

----- Epoch 5/10 -----
Train Loss: 0.3178 | Train Acc: 90.36%
Train Loss: 0.3178 | Train Acc: 90.36%
Val Loss: 0.1091 | Val Acc: 96.92%

----- Epoch 6/10 -----
Val Loss: 0.1091 | 